# Problem 3: Autocomplete Application
### Left-to-Right Complete + Overall Complete
**Approach:** Hybrid N-gram + LSTM model on human conversation data

- **Left-to-Right**: Given partial input, predict the next word(s) one at a time
- **Overall Complete**: Given partial input, generate a full sentence continuation
- **Interface Design**: Context-aware, works mid-sentence or after finishing typing

## 1. Environment Setup

In [ ]:
# Mount Google Drive if using Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/miniproject')
    print('Mounted Google Drive')
except:
    print('Not running in Colab — skipping drive mount')

In [ ]:
import re, json, random, pickle
import numpy as np
import nltk
from collections import defaultdict, Counter
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Libraries loaded ✅')

## 2. Load & Preprocess Data

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
FILE_PATH = 'human_chat.txt'   # adjust if needed

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f'Loaded {len(raw_text):,} characters')
print(raw_text[:300])

In [ ]:
# ── Preprocess ────────────────────────────────────────────────────────────────
def preprocess(text):
    """Clean text and return list of sentences, each a list of tokens."""
    # Remove speaker labels like 'Human 1:'
    text = re.sub(r'Human \d+:', '', text)
    # Lowercase
    text = text.lower()
    # Keep letters, spaces, apostrophes
    text = re.sub(r"[^a-z\s']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize into sentences then words
    sentences = nltk.sent_tokenize(text)
    tokenized = [nltk.word_tokenize(s) for s in sentences if len(s.split()) > 2]
    return tokenized

sentences = preprocess(raw_text)
all_words  = [w for s in sentences for w in s]
print(f'Sentences: {len(sentences):,}')
print(f'Total tokens: {len(all_words):,}')
print(f'Sample sentence: {sentences[0]}')

In [ ]:
# ── Vocabulary ────────────────────────────────────────────────────────────────
MIN_FREQ = 2  # drop very rare words

freq = Counter(all_words)
vocab = ['<UNK>', '<S>', '</S>'] + sorted(w for w,c in freq.items() if c >= MIN_FREQ)
vocab_size = len(vocab)
word_to_int = {w: i for i, w in enumerate(vocab)}
int_to_word = {i: w for i, w in enumerate(vocab)}

def w2i(w): return word_to_int.get(w, 0)  # 0 = <UNK>

print(f'Vocabulary size (min_freq={MIN_FREQ}): {vocab_size:,}')

## 3. N-gram Language Model
We build bigram and trigram models. These work well on small datasets and are used as the primary inference engine. The LSTM (Part 4) is also trained for the neural approach.

In [ ]:
# ── Bigram & Trigram counts ────────────────────────────────────────────────────
bigram_counts  = defaultdict(Counter)
trigram_counts = defaultdict(Counter)

for sent in sentences:
    tokens = ['<S>'] + [w if freq[w] >= MIN_FREQ else '<UNK>' for w in sent] + ['</S>']
    for i in range(len(tokens) - 1):
        bigram_counts[tokens[i]][tokens[i+1]] += 1
    for i in range(len(tokens) - 2):
        trigram_counts[(tokens[i], tokens[i+1])][tokens[i+2]] += 1

print(f'Unique bigram contexts:  {len(bigram_counts):,}')
print(f'Unique trigram contexts: {len(trigram_counts):,}')

In [ ]:
# ── Add-lambda smoothed probability ───────────────────────────────────────────
LAMBDA = 0.1  # tuned on dev set (see Problem 1 for formal tuning)

def bigram_prob(w_prev, w):
    numer = bigram_counts[w_prev][w] + LAMBDA
    denom = sum(bigram_counts[w_prev].values()) + LAMBDA * vocab_size
    return numer / denom

def trigram_prob(w1, w2, w):
    ctx = (w1, w2)
    numer = trigram_counts[ctx][w] + LAMBDA
    denom = sum(trigram_counts[ctx].values()) + LAMBDA * vocab_size
    return numer / denom

# Interpolated probability: alpha * trigram + (1-alpha) * bigram
ALPHA = 0.6

def interp_prob(w1, w2, w):
    return ALPHA * trigram_prob(w1, w2, w) + (1 - ALPHA) * bigram_prob(w2, w)

print('N-gram model built ✅')

## 4. LSTM Language Model (Neural)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# ── Prepare sequences ─────────────────────────────────────────────────────────
# Use a SHORTER window for small dataset — 10 words is enough
SEQ_LEN = 10

int_sentences = []
for sent in sentences:
    int_sentences.append([w2i(w) for w in sent])

X_list, y_list = [], []
for sent_ids in int_sentences:
    padded = [w2i('<S>')] * SEQ_LEN + sent_ids + [w2i('</S>')]
    for i in range(len(padded) - SEQ_LEN):
        X_list.append(padded[i:i+SEQ_LEN])
        y_list.append(padded[i+SEQ_LEN])

X = np.array(X_list)
y = to_categorical(y_list, num_classes=vocab_size)
print(f'Training samples: {X.shape[0]:,}, vocab: {vocab_size}')

In [ ]:
# ── Build model ───────────────────────────────────────────────────────────────
# Simpler architecture — better for small dataset
model = Sequential([
    Embedding(vocab_size, 64, input_length=SEQ_LEN),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(128),
    Dropout(0.3),
    Dense(vocab_size, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
# Split: 80% train, 20% val
history = model.fit(
    X, y,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)
    ]
)
print('LSTM training complete ✅')

In [ ]:
# ── Save model & artifacts ─────────────────────────────────────────────────────
model.save('email_autocomplete_model_v2.keras')

artifacts = {
    'sequence_length': SEQ_LEN,
    'vocab_size': vocab_size,
    'word_to_int': word_to_int,
    'int_to_word': {str(k): v for k, v in int_to_word.items()},
    'lambda': LAMBDA,
    'alpha': ALPHA
}
with open('artifacts_v2.json', 'w') as f:
    json.dump(artifacts, f)

# Save n-gram models
with open('ngram_models.pkl', 'wb') as f:
    pickle.dump({'bigram': dict(bigram_counts), 'trigram': dict(trigram_counts)}, f)

print('Saved all artifacts ✅')

## 5. Autocomplete Functions

### Design Decision:
- **Left-to-Right**: User types words → system suggests the **top-k next words** in real time
- **Overall Complete**: User provides partial sentence → system generates **full sentence** until `</S>` or max length
- **Context**: Both modes use the last N words as context (with-context interface)
- **Backend**: Interpolated N-gram (fast, reliable on small data) + LSTM option

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  AUTOCOMPLETE CORE FUNCTIONS
# ═══════════════════════════════════════════════════════════════

def tokenize_input(text):
    """Clean and tokenize user input."""
    text = text.lower()
    text = re.sub(r"[^a-z\s']", ' ', text)
    return nltk.word_tokenize(text.strip())


def get_top_k_ngram(context_tokens, k=5, temperature=1.0):
    """
    LEFT-TO-RIGHT: Given a list of context tokens, return top-k next word predictions
    using interpolated trigram + bigram model.
    
    Args:
        context_tokens: list of words already typed
        k: number of suggestions to return
        temperature: >1 = more diverse, <1 = more conservative
    Returns:
        List of (word, probability) tuples, sorted by probability desc
    """
    if not context_tokens:
        context_tokens = ['<S>']
    
    # Map to known vocab
    ctx = [w if w in word_to_int else '<UNK>' for w in context_tokens]
    w1 = ctx[-2] if len(ctx) >= 2 else '<S>'
    w2 = ctx[-1]
    
    # Score every vocab word
    scores = {}
    for w in vocab:
        if w in ('<S>', '<UNK>'):  # skip special tokens
            continue
        scores[w] = interp_prob(w1, w2, w)
    
    # Apply temperature
    if temperature != 1.0:
        scores = {w: p ** (1.0 / temperature) for w, p in scores.items()}
    
    # Sort and return top-k
    top_k = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return top_k


def left_to_right_complete(partial_text, n_suggestions=5, temperature=0.8):
    """
    LEFT-TO-RIGHT AUTOCOMPLETE:
    Given partial text, suggest the next word(s).
    
    Returns:
        dict with 'next_word_suggestions' (list of words)
    """
    tokens = tokenize_input(partial_text)
    if not tokens:
        return {'input': partial_text, 'next_word_suggestions': []}
    
    suggestions = get_top_k_ngram(tokens, k=n_suggestions, temperature=temperature)
    
    return {
        'input': partial_text,
        'next_word_suggestions': [w for w, p in suggestions],
        'probabilities': [round(p, 4) for w, p in suggestions]
    }


def overall_complete(partial_text, max_words=30, temperature=0.9, strategy='sampling'):
    """
    OVERALL COMPLETE:
    Given partial text, generate a full sentence continuation.
    
    Args:
        partial_text: what the user has typed so far
        max_words: maximum words to generate
        temperature: sampling diversity
        strategy: 'greedy' (most likely) or 'sampling' (random weighted)
    Returns:
        dict with 'completion' (just the generated part) and 'full_text'
    """
    tokens = tokenize_input(partial_text)
    if not tokens:
        return {'input': partial_text, 'completion': '', 'full_text': partial_text}
    
    generated = []
    current_tokens = list(tokens)
    
    for _ in range(max_words):
        candidates = get_top_k_ngram(current_tokens, k=20, temperature=temperature)
        
        if not candidates:
            break
        
        if strategy == 'greedy':
            next_word = candidates[0][0]
        else:  # sampling — weighted random from top candidates
            words, probs = zip(*candidates)
            total = sum(probs)
            norm_probs = [p / total for p in probs]
            next_word = random.choices(words, weights=norm_probs, k=1)[0]
        
        if next_word == '</S>':
            break
        
        generated.append(next_word)
        current_tokens.append(next_word)
        # Keep context window manageable
        if len(current_tokens) > 10:
            current_tokens = current_tokens[-10:]
    
    completion = ' '.join(generated)
    full_text   = partial_text.rstrip() + (' ' if partial_text else '') + completion
    
    return {
        'input': partial_text,
        'completion': completion,
        'full_text': full_text
    }


# ── LSTM-based prediction (for comparison) ────────────────────────────────────
def lstm_predict_next(partial_text, k=5, temperature=1.0):
    """
    Use the trained LSTM to predict next word.
    Works best when model is well-trained; n-gram is the fallback.
    """
    tokens = tokenize_input(partial_text)
    ids = [w2i(w) for w in tokens]
    
    # Pad/truncate to SEQ_LEN
    if len(ids) < SEQ_LEN:
        ids = [w2i('<S>')] * (SEQ_LEN - len(ids)) + ids
    else:
        ids = ids[-SEQ_LEN:]
    
    x = np.array([ids])
    probs = model.predict(x, verbose=0)[0]
    
    # Apply temperature
    probs = np.log(probs + 1e-10) / temperature
    probs = np.exp(probs) / np.sum(np.exp(probs))
    
    top_k_ids = np.argsort(probs)[::-1][:k]
    return [(int_to_word[i], float(probs[i])) for i in top_k_ids
            if int_to_word[i] not in ('<S>', '<UNK>', '</S>')]


print('Autocomplete functions defined ✅')

## 6. Demo: Left-to-Right Complete

In [ ]:
# ── Left-to-Right Examples ────────────────────────────────────────────────────
test_inputs_ltr = [
    "Hi I hope you are",
    "I wanted to follow up",
    "Could you please let me",
    "Thank you for your",
    "I think it would be"
]

print('═' * 60)
print('LEFT-TO-RIGHT AUTOCOMPLETE — Top 5 Next Word Suggestions')
print('═' * 60)
for text in test_inputs_ltr:
    result = left_to_right_complete(text, n_suggestions=5)
    print(f'\nInput: "{text}"')
    print(f'Suggestions: {result["next_word_suggestions"]}')

## 7. Demo: Overall Complete

In [ ]:
# ── Overall Complete Examples ─────────────────────────────────────────────────
test_inputs_oc = [
    "Hi I hope you are",
    "I wanted to follow up about",
    "Could you please",
    "I think",
    "Do you"
]

print('═' * 60)
print('OVERALL COMPLETE — Full Sentence Generation')
print('═' * 60)
for text in test_inputs_oc:
    result = overall_complete(text, max_words=20, strategy='sampling')
    print(f'\nInput:      "{text}"')
    print(f'Completion: "{result["completion"]}"')
    print(f'Full text:  "{result["full_text"]}"')

## 8. Greedy vs Sampling Comparison

In [ ]:
seed = "I think it would be good if we"
print(f'Seed: "{seed}"\n')

print('--- GREEDY (deterministic, most probable) ---')
for _ in range(3):
    r = overall_complete(seed, max_words=15, strategy='greedy')
    print(f'  {r["full_text"]}')

print('\n--- SAMPLING (diverse, temperature=0.9) ---')
for _ in range(5):
    r = overall_complete(seed, max_words=15, strategy='sampling', temperature=0.9)
    print(f'  {r["full_text"]}')

## 9. LSTM vs N-gram Comparison

In [ ]:
test = "I hope you are doing"

print(f'Input: "{test}"\n')
print('N-gram top-5:', [w for w,p in get_top_k_ngram(tokenize_input(test), k=5)])
print('LSTM   top-5:', [w for w,p in lstm_predict_next(test, k=5)])

## 10. Streamlit App
The `app.py` file below contains the full Streamlit demo. Run it with:
```bash
streamlit run app.py
```
See the `app.py` file for the complete interface with left-to-right and overall complete buttons.